# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ssaqlain09/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane:** Refresh / Content Opportunity Scoring.

**Task type: ranking (via scoring).** The question is *"which pages should a reviewer look at first?"* —
a "which ones first?" question, not a "yes/no for every page" question. The output is a ranked review queue.
Under the hood I will score each page with a classifier (probability that the page is declining) and sort
by that score, so the model is a classifier but the *product* is a ranked list.

**One-paragraph frame:** For a content editor with limited weekly review capacity, deciding which pages to
review first, I will build a ranked review queue from the 30,000-row starter dataset, scoring each page's
decline risk, measured by Precision@K (compared with the base rate and a simple hand rule). A wrong call
costs wasted reviewer hours (false positive) or a silently declining page that never gets looked at
(false negative). A plain rule is not enough because the decline pattern depends on several signals at
once (see section 5). I will claim only observed / directional / decision-support results.

In [1]:
# Task-type check: a ranking output means I care about the TOP of the list, not every row.
REVIEW_CAPACITY_K = 50   # assumption: a reviewer can act on ~50 pages per cycle
print(f"Task: ranking/scoring | review capacity assumed: top {REVIEW_CAPACITY_K} pages")

Task: ranking/scoring | review capacity assumed: top 50 pages


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (starter data): `is_declining_label`** = 1 when `trend_direction == "down"`, else 0.

**This is a proxy, not the ideal target — and I want to be honest about why.** The label is a *bucket
computed from the current 90-day window* (`trend_direction`, itself derived from `trend_pct`). It is
a defined rule, not a future outcome that I observed later. Two consequences:

- `trend_direction` and `trend_pct` can **never be used as features** — they *are* the label in disguise
  (leakage). I will exclude them.
- The stronger version of this project (on the warehouse data) would use a **future-window label**:
  features from the prior 90 days → did impressions decline over the *next* 30 days? That is an observed
  outcome. I will treat the starter label as a practice proxy and say so in any claim I make.

Also excluded as features: `content_id` and `client_id` (pseudonymous IDs — used only for grouping and
for client-holdout splits).

In [2]:
import os
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Sketch of the target column (label derived from trend_direction):")
print(df["trend_direction"].value_counts().to_string())
print()
print(f"Positive (declining) rows: {df['is_declining_label'].sum():,} of {len(df):,}")
print(f"Base rate: {df['is_declining_label'].mean():.3f}")

# Columns that must NOT be features (label leakage or pseudonymous IDs)
NEVER_FEATURES = ["trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"]
print("\nExcluded from features:", NEVER_FEATURES)

Sketch of the target column (label derived from trend_direction):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Positive (declining) rows: 16,262 of 30,000
Base rate: 0.542

Excluded from features: ['trend_direction', 'trend_pct', 'is_declining_label', 'content_id', 'client_id']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50** — of the 50 pages the system says to review first, how many are actually
positive on the chosen label? I pick K = 50 because it matches the assumed review capacity (a reviewer
can realistically act on ~50 pages per cycle). Ranked queues are used from the top, so top-K precision
matches how the list is actually used better than overall accuracy.

**What "good" means:** Precision@50 must beat **two** reference points, evaluated with a **client-holdout**
split (whole clients kept out of training, so the model is tested on clients it never saw):

1. the **base rate** of the test set I use. Across all 30,000 rows it is 0.542. The starter pipeline's
   held-out test clients (6 clients, 2,325 rows) have a lower base rate, about 0.391 — so always compare
   Precision@50 with the base rate of the *same* rows it was measured on, and
2. the **hand-written baseline rule** (Precision@50 = 0.240 in the starter pipeline).

Reference from the lane guide: the starter random forest reached Precision@50 = 0.740 versus 0.240 for the
baseline rules. I read that as promising on this slice, not proof — it is a small 30,000-row sample, and
the label is a proxy. I will also watch recall and average precision so I don't over-trust one number.

In [3]:
# What the metric looks like on the label: Precision@K for a ranked list = mean of the label in the top K.
import numpy as np

def precision_at_k(scores, labels, k):
    order = scores.sort_values(ascending=False).index[:k]
    return labels.loc[order].mean()

# Demo only: a random ranking should land near the base rate (this is the floor to beat).
# A single draw of 50 pages is noisy, so average many random draws.
rng = np.random.default_rng(42)
draws = [precision_at_k(pd.Series(rng.random(len(df)), index=df.index), df["is_declining_label"], 50)
         for _ in range(200)]
print(f"Random-ranking Precision@50 (mean of 200 draws): {np.mean(draws):.3f}  (std {np.std(draws):.3f})")
print(f"Base rate:                                      {df['is_declining_label'].mean():.3f}")


Random-ranking Precision@50 (mean of 200 draws): 0.534  (std 0.064)
Base rate:                                      0.542


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page (one `content_id`, belonging to one client), summarised over a trailing
90-day window.** The starter file has 30,000 such rows across 32 clients. Because rows from the same
client can share patterns, I will split by `client_id` when validating.

In [4]:
lane_cols = [
    "content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "position_tier", "content_age_days",
    "days_since_last_update", "freshness_tier", "trend_direction", "is_declining_label",
]
unit = df[lane_cols]
print(f"Rows (pages): {len(unit):,} | Distinct clients: {unit['client_id'].nunique()}")
print(f"Duplicate content_id rows: {unit['content_id'].duplicated().sum()}  (1 row = 1 page)")
unit.head(8)

Rows (pages): 30,000 | Distinct clients: 32
Duplicate content_id rows: 0  (1 row = 1 page)


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,position_tier,content_age_days,days_since_last_update,freshness_tier,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,striking,187,20,0-30,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,page_3_5,445,25,0-30,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,page_3_5,141,20,0-30,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,page_1,463,22,0-30,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,page_3_5,263,14,0-30,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,0.03,8.5,page_1,147,20,0-30,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,0.00,7.0,page_1,90,20,0-30,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,0.06,21.2,page_3_5,445,22,0-30,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "stale and visible = refresh" looks at one or two signals. The table below shows the
decline rate is **not** driven by a single signal: it moves with position tier, with content type, and
differs across clients — and these interact. That is the kind of "real but tangled" pattern where a
learned model can earn its place (many signals, weak individually, useful together).

Evidence from the starter pipeline (lane guide): the hand-written rules scored Precision@50 = 0.240 while
the random forest scored 0.740 under client-holdout. Notably the hand rule's 0.240 is *below* even the ~0.391 base rate
of that pipeline's held-out test clients (6 clients, 2,325 rows), meaning its top picks were worse than
random on that test slice. The random forest's 0.740 is roughly 1.9x that same base rate. A test set of only
6 clients is small, so I treat these as directional, not settled. **Caveat:** these are differences
I observed on one 30,000-row slice with a proxy label; they show a model can rank better than that
particular rule here, not that ML always beats rules. If a simple rule matched the model, I should use the
rule — it is easier to explain.

**Content action the output supports:** each ranked page maps to a reviewer action — refresh, fix
title/meta if CTR is low for its position tier, expand thin content, or just monitor.

In [5]:
# Decline rate is not a one-signal story: it varies by position tier, content type, and client.
def rate_table(col):
    t = df.groupby(col)["is_declining_label"].agg(rate="mean", n="size").round(3)
    return t.sort_values("rate")

print("Decline rate by position_tier:"); print(rate_table("position_tier").to_string()); print()
print("Decline rate by content_type:");  print(rate_table("content_type").to_string()); print()

client_rate = df.groupby("client_id")["is_declining_label"].mean()
print("Decline rate across the 32 clients:")
print(client_rate.describe().round(3).to_string())

Decline rate by position_tier:
                rate      n
position_tier              
top_3          0.241   2321
deep           0.344   1319
page_3_5       0.562   7242
page_1         0.570  11814
striking       0.610   7304

Decline rate by content_type:
                     rate      n
content_type                    
feedly article      0.287   2096
keyword article     0.561  27207
comparison article  0.572    697

Decline rate across the 32 clients:
count    32.000
mean      0.489
std       0.227
min       0.000
25%       0.410
50%       0.524
75%       0.625
max       0.937


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.